In [1]:
import os
os.environ["OPENAI_API_KEY"]=""


In [2]:
from __future__ import annotations

import json
import os
import re
import subprocess
import tempfile
from dataclasses import dataclass, asdict
from typing import Any, Dict, Optional

from phi.agent import Agent
from phi.model.openai import OpenAIChat


# ----------------------------
# Data structures
# ----------------------------

@dataclass
class RunResult:
    ok: bool
    exit_code: int
    stdout: str
    stderr: str
    wall_time_ms: int


@dataclass
class EvaluationResult:
    score_0_to_10: int
    summary: str
    issues: list[str]
    improvements: list[str]


# ----------------------------
# Tool 2: Run code safely-ish
# ----------------------------

def run_python_code(code: str, timeout_s: int = 10) -> Dict[str, Any]:
    """
    Runs python code in a temp file with a timeout.
    Captures stdout/stderr and returns a dict RunResult-like payload.

    NOTE: This is not a hardened sandbox. For untrusted code, use real sandboxing
    (containers, seccomp, firejail, etc).
    """
    import time

    start = time.time()
    with tempfile.TemporaryDirectory() as td:
        path = os.path.join(td, "main.py")
        with open(path, "w", encoding="utf-8") as f:
            f.write(code)

        try:
            proc = subprocess.run(
                ["python", path],
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )
            end = time.time()
            result = RunResult(
                ok=(proc.returncode == 0),
                exit_code=proc.returncode,
                stdout=proc.stdout or "",
                stderr=proc.stderr or "",
                wall_time_ms=int((end - start) * 1000),
            )
            return asdict(result)

        except subprocess.TimeoutExpired as e:
            end = time.time()
            result = RunResult(
                ok=False,
                exit_code=124,
                stdout=e.stdout or "",
                stderr=(e.stderr or "") + "\nTIMEOUT",
                wall_time_ms=int((end - start) * 1000),
            )
            return asdict(result)


# ----------------------------
# Tool 1: Code generator agent
# ----------------------------

def make_code_writer(model: Optional[Any] = None) -> Agent:
    return Agent(
        name="CodeWriter",
        model=model or OpenAIChat(id="gpt-5-nano"),
        description="You write correct, minimal Python code for the given task.",
        instructions=[
            "Return ONLY raw, executable Python source code.",
    "Do NOT wrap the code in objects, variables, JSON, or metadata (e.g., no content=..., no RunResponse, no Message objects).",
    "Do NOT include markdown fences, explanations, comments about the response format, or any extra text.",
    "Output must start directly with valid Python syntax (e.g., import, def, class, or if __name__ == '__main__':).",
    "Follow best coding practices (clear naming, type hints where appropriate, docstrings when useful).",
    "Prefer the Python standard library unless third-party libraries are explicitly required.",
    "Include a main guard when appropriate: if __name__ == '__main__':",
    "Ensure the code is directly runnable as a standalone .py file."
        ],
        markdown=False,
    )


def generate_python_code1(task: str, writer: Agent) -> str:
    code = writer.run(task, stream=False)
    # Defensive cleanup in case a model returns fenced blocks anyway.
    code = re.sub(r"^```(?:python)?\s*", "", str(code).strip(), flags=re.IGNORECASE)
    code = re.sub(r"\s*```$", "", code.strip())
    #print(code)
    return code.strip()

def generate_python_code(task: str, writer: Agent) -> str:
    resp = writer.run(task, stream=False)

    # 1) If the framework returned a structured object with .content, prefer that.
    try:
        content = getattr(resp, "content", None)
        if isinstance(content, str) and content.strip():
            text = content
        else:
            text = str(resp)
    except Exception:
        text = str(resp)

    text = text.strip()

    # Helper: remove markdown code fences if present
    def _strip_fences(s: str) -> str:
        s = re.sub(r"^\s*```(?:python)?\s*", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s*```\s*$", "", s)
        return s.strip()

    # Helper: unescape if the code is inside a quoted python string repr
    def _unescape_python_string_literal(s: str) -> str:
        # Turn sequences like \\n, \\' into real newlines and quotes.
        # (unicode_escape is appropriate for backslash escapes from repr-like payloads)
        try:
            return bytes(s, "utf-8").decode("unicode_escape")
        except Exception:
            return s

    # 2) If the string contains the assistant message block, extract it.
    # Matches: Message(role='assistant', content='...') or content="..."
    m = re.search(
        r"Message\(\s*role\s*=\s*['\"]assistant['\"].*?content\s*=\s*(?P<q>['\"])(?P<body>.*?)(?P=q)\s*[,\)]",
        text,
        flags=re.DOTALL,
    )
    if m:
        extracted = m.group("body")
        extracted = _unescape_python_string_literal(extracted)
        extracted = _strip_fences(extracted)
        if extracted:
            return extracted

    # 3) Extract a top-level content='...'
    # Matches: content='...'\s+content_type='str' event='RunResponse' ...
    m = re.search(
        r"\bcontent\s*=\s*(?P<q>['\"])(?P<body>.*?)(?P=q)\s+\bcontent_type\b",
        text,
        flags=re.DOTALL,
    )
    if m:
        extracted = m.group("body")
        extracted = _unescape_python_string_literal(extracted)
        extracted = _strip_fences(extracted)
        if extracted:
            return extracted

    # 4) Last resort: remove fences and return whatever remains.
    text = _strip_fences(text)

    # If it still begins with "content='", try a simpler parse.
    if text.startswith("content="):
        m = re.search(r"^content\s*=\s*(?P<q>['\"])(?P<body>.*?)(?P=q)\s*$", text, flags=re.DOTALL)
        if m:
            extracted = _unescape_python_string_literal(m.group("body"))
            extracted = _strip_fences(extracted)
            if extracted:
                return extracted

    return text

# ----------------------------
# Agent: Code reviewer
# ----------------------------

def make_code_reviewer(model: Optional[Any] = None) -> Agent:
    return Agent(
        name="CodeReviewer",
        model=model or OpenAIChat(id="gpt-5-nano"),
        description="You review Python code using pylint-style feedback.",
        instructions=[
            "You are a strict Python code reviewer.",
            "Review the provided Python code.",
            "Explain issues clearly and suggest improvements.",
            "Focus on correctness, readability, typing, structure, and best practices.",
            "Do NOT rewrite the entire code unless necessary.",
            "provde only the score of the analysis between 1 and 10, 10 being the best score of the code",
        ],
        markdown=False,
    )
# ----------------------------
# Tool 3: Static code analysis with pylint
# ----------------------------

def run_pylint_analysis1(code: str, timeout_s: int = 20) -> Dict[str, Any]:
    """
    Runs pylint static analysis on the provided Python code.
    Returns structured JSON results including score and issues.

    Requires pylint installed in the environment.
    """
    import time

    start = time.time()

    with tempfile.TemporaryDirectory() as td:
        path = os.path.join(td, "main.py")
        with open(path, "w", encoding="utf-8") as f:
            f.write(code)

        try:
            proc = subprocess.run(
                [
                    "pylint",
                    path,
                    "--output-format=json",
                    "--score=y",
                ],
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )

            end = time.time()

            try:
                issues = json.loads(proc.stdout) if proc.stdout.strip() else []
            except json.JSONDecodeError:
                issues = []

            result = {
                "ok": proc.returncode == 0,
                "exit_code": proc.returncode,
                "issues": issues,
                "stderr": proc.stderr or "",
                "wall_time_ms": int((end - start) * 1000),
            }

            return result

        except subprocess.TimeoutExpired as e:
            end = time.time()
            return {
                "ok": False,
                "exit_code": 124,
                "issues": [],
                "stderr": (e.stderr or "") + "\nPYLINT TIMEOUT",
                "wall_time_ms": int((end - start) * 1000),
            }

def run_pylint_analysis(code: str, timeout_s: int = 20) -> Dict[str, Any]:
    """
    Runs pylint static analysis on the provided Python code.

    Returns a high-level summary:
    - count_convention
    - count_refactor
    - count_warning
    - count_error
    - count_fatal
    - total_issues
    - score (if available)

    Requires pylint installed.
    """
    import time

    start = time.time()

    with tempfile.TemporaryDirectory() as td:
        path = os.path.join(td, "main.py")
        with open(path, "w", encoding="utf-8") as f:
            f.write(code)

        try:
            proc = subprocess.run(
                [
                    "pylint",
                    path,
                    "--output-format=json",
                    "--score=y",
                ],
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )

            end = time.time()

            # -----------------------
            # Parse issues JSON
            # -----------------------
            try:
                issues = json.loads(proc.stdout) if proc.stdout.strip() else []
            except json.JSONDecodeError:
                issues = []

            summary = {
                "convention": 0,
                "refactor": 0,
                "warning": 0,
                "error": 0,
                "fatal": 0,
            }

            for issue in issues:
                issue_type = issue.get("type", "").lower()
                if issue_type in summary:
                    summary[issue_type] += 1

            total_issues = sum(summary.values())

            # -----------------------
            # Extract score
            # -----------------------
            score = None
            score_match = re.search(
                r"rated at\s+([-+]?\d+\.\d+)/10",
                proc.stderr or "",
            )
            if not score_match:
                score_match = re.search(
                    r"rated at\s+([-+]?\d+\.\d+)/10",
                    proc.stdout or "",
                )

            if score_match:
                try:
                    score = float(score_match.group(1))
                except ValueError:
                    score = None

            return {
                "ok": proc.returncode == 0,
                "exit_code": proc.returncode,
                "score": score,
                "total_issues": total_issues,
                "count_convention": summary["convention"],
                "count_refactor": summary["refactor"],
                "count_warning": summary["warning"],
                "count_error": summary["error"],
                "count_fatal": summary["fatal"],
                "wall_time_ms": int((end - start) * 1000),
            }

        except subprocess.TimeoutExpired as e:
            end = time.time()
            return {
                "ok": False,
                "exit_code": 124,
                "score": None,
                "total_issues": 0,
                "count_convention": 0,
                "count_refactor": 0,
                "count_warning": 0,
                "count_error": 0,
                "count_fatal": 0,
                "wall_time_ms": int((end - start) * 1000),
                "stderr": (e.stderr or "") + "\nPYLINT TIMEOUT",
            }


In [5]:
import sys
writer = make_code_writer()
reviewer = make_code_reviewer()

#task = "Write a function that computes fibonacci recursively till 100."
task1="""
The proper divisors of a number are all the divisors excluding the number itself. For example, the proper divisors of 2828 are 11, 22, 44, 77, and 1414. As the sum of these divisors is equal to 2828, we call it a perfect number.

Interestingly the sum of the proper divisors of 220220 is 284284 and the sum of the proper divisors of 284284 is 220220, forming a chain of two numbers. For this reason, 220220 and 284284 are called an amicable pair.

Perhaps less well known are longer chains. For example, starting with 1249612496, we form a chain of five numbers:
12496→14288→15472→14536→14264(→12496→⋯ )
12496→14288→15472→14536→14264(→12496→⋯)

Since this chain returns to its starting point, it is called an amicable chain.

Find the smallest member of the longest amicable chain with no element exceeding one million.
"""
task="Euler problem 66"
code = generate_python_code(task, writer)
print(code)

run_results=run_python_code(code)
print(run_results)



analysis = run_pylint_analysis(code)
print(analysis)
# sys.exit(0)
review_feedback = reviewer.run(
    f"Here is the code:\n\n{code}\n\nHere are pylint issues:\n\n{json.dumps(analysis, indent=2)}"
)

print(review_feedback.content)

import math

def fundamental_pell(n: int):
    a0 = int(math.isqrt(n))
    if a0 * a0 == n:
        return None
    m = 0
    d = 1
    a = a0
    period = []
    while True:
        m = d * a - m
        d = (n - m * m) // d
        a = (a0 + m) // d
        period.append(a)
        if a == 2 * a0:
            break
    L = len(period)
    if L % 2 == 0:
        needed = L - 1
        a_seq = [a0] + period[:needed]
    else:
        needed = 2 * L - 1
        repeats = (needed + L - 1) // L
        a_seq = [a0] + (period * repeats)[:needed]
    p_minus2, p_minus1 = 0, 1
    q_minus2, q_minus1 = 1, 0
    for ak in a_seq:
        p = ak * p_minus1 + p_minus2
        q = ak * q_minus1 + q_minus2
        p_minus2, p_minus1 = p_minus1, p
        q_minus2, q_minus1 = q_minus1, q
    return p, q

def main():
    best_n = None
    max_x = -1
    for n in range(2, 1001):
        res = fundamental_pell(n)
        if res is None:
            continue
        x, y = res
        if x > max_x:
    